# Module 4 lab · Your first language model

The code from every lesson of [this module](https://aicourses.dev/research/first-model), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 4.2 Getting a ready-made model: what you download and how to run it

Trained models are shared online for anyone to download. A model is a file of learned numbers plus the code that uses them, and a few lines of a library called Hugging Face Transformers load it and run it.

[Read the lesson](https://aicourses.dev/research/first-model/ready-made-models)

### Loading it

Then three lines load the model:

The lesson installs this first:

In [ ]:
!pip install -U transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

Count the parameters to check you have the model you think you have:

In [ ]:
n = sum(p.numel() for p in model.parameters())
print(f"{n:,}")

### Text in, numbers out

A model never sees letters. The tokenizer turns text into token ids, whole numbers that point into the vocabulary:

In [ ]:
ids = tokenizer("Hello, world")["input_ids"]
print(ids)
print([tokenizer.decode(i) for i in ids])

### Making it write

Now give it a start and let it continue:

In [ ]:
text = "The best thing about rainy days is"
inputs = tokenizer(text, return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=20,
                     do_sample=False)
print(tokenizer.decode(out[0]))

## 4.3 What a language model actually does

You run a small real model and print what it gives each possible next token: a raw score called a logit, which becomes a probability. You then pick tokens at random by those chances, and a setting called temperature makes the picks bolder or safer.

[Read the lesson](https://aicourses.dev/research/first-model/what-a-language-model-does)

### One run of the model: a score for every token

Load GPT-2 as in lesson 4.2. Then give it some text and run the model once, without `generate`:

In [ ]:
import numpy as np
import torch

text = "The capital of France is"
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():          # just running it, not training
    out = model(**inputs)
print(out.logits.shape)

You want the guess for what comes after the whole text, which is the last row:

In [ ]:
logits = out.logits[0, -1].double().numpy()   # last row

### From scores to chances

In code, with one safety step:

In [ ]:
def probs(logits, temperature=1.0):
    z = logits / temperature
    p = np.exp(z - z.max())     # subtract the max: no overflow
    return p / p.sum()

Now print the ten likeliest next tokens:

In [ ]:
p = probs(logits)
for i in np.argsort(p)[::-1][:10]:     # biggest first
    print(repr(tokenizer.decode(int(i))), round(p[i], 3))

### Temperature: safer or bolder picks

Here is the same toy example at three temperatures:

In [ ]:
toy = np.array([2.0, 1.0, 0.0])     # three made-up logits
for t in [0.5, 1.0, 2.0]:
    print(t, probs(toy, t).round(3))
# 0.5 [0.867 0.117 0.016]
# 1.0 [0.665 0.245 0.09 ]
# 2.0 [0.506 0.307 0.186]

### The loop, written by you

Now put it together: score, turn into chances, pick, add, repeat. This writes 20 tokens at temperature 0.8:

In [ ]:
rng = np.random.default_rng(seed=0)
ids = inputs["input_ids"]
for _ in range(20):
    with torch.no_grad():
        logits = model(ids).logits[0, -1].double().numpy()
    next_id = int(rng.choice(len(logits), p=probs(logits, 0.8)))
    ids = torch.cat([ids, torch.tensor([[next_id]])], dim=1)
print(tokenizer.decode(ids[0]))

## 4.5 Your lab bench: Colab, seeds, a tracker and a research notebook

A seed is the number that fixes a program’s random choices, so a rerun gives the same result. Seeds, a tool that records each run’s settings and results, and a dated research notebook make every later result one you can repeat.

[Read the lesson](https://aicourses.dev/research/first-model/lab-bench)

### Where to run

Check which kind of runtime you have:

In [ ]:
import torch

print(torch.cuda.is_available())   # an NVIDIA GPU?

### Seeds: making chance repeatable

You met seeds in lesson 3.6. Here they are again, for NumPy and for PyTorch:

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)
print(rng.integers(1, 7, size=5))   # [1 5 4 3 3]
rng = np.random.default_rng(seed=42)
print(rng.integers(1, 7, size=5))   # [1 5 4 3 3]

torch.manual_seed(42)
print(torch.rand(3))   # tensor([0.8823, 0.9150, 0.3829])
torch.manual_seed(42)
print(torch.rand(3))   # tensor([0.8823, 0.9150, 0.3829])

### A tracker: every run, written down

A **tracker** records, for every run, the settings that went in and the numbers that came out. The simplest one is a file you add a line to:

In [ ]:
import json, datetime

run = {
    "date": datetime.date.today().isoformat(),
    "model": "openai-community/gpt2",
    "prompt": "The capital of France is",
    "seed": 42,
    "temperature": 0.8,
}
with open("runs.jsonl", "a") as f:    # "a" adds to the end
    f.write(json.dumps(run) + "\n")

A file works, but a real tracker also draws your numbers as charts and lets you compare runs. **Trackio**, a free tracker from Hugging Face, needs three calls. Install it first, in a cell of its own: `!pip install trackio`.

The lesson installs this first:

In [ ]:
!pip install trackio

In [ ]:
import trackio

trackio.init(project="first-model",
             config={"seed": 42, "temperature": 0.8})
trackio.log({"top_prob": float(p.max())})   # p from lesson 4.3
trackio.finish()

## 4.6 Project: first-model notebook

Your first entry in the portfolio repo: a real model, run, inspected, sampled and logged so anyone can rerun it.

[Read the lesson](https://aicourses.dev/research/first-model/project-first-model-notebook)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Steps

1. **Set up.** Open a new Colab notebook on the CPU runtime. The first code cell installs what you need with `!pip install -U transformers trackio`, the next imports them along with NumPy and PyTorch, prints `transformers.__version__` and `torch.__version__`, and sets one seed with `torch.manual_seed(0)` and `rng = np.random.default_rng(seed=0)`.
2. **Load the model.** Load `openai-community/gpt2` and its tokenizer as in lesson 4.2, and print its parameter count.
3. **Choose five prompts.** Make them different kinds: a fact (“The capital of Italy is”), something open (“My favourite thing to cook is”), a list, a line of code and an instruction. Before running anything, write your guess for each one’s top token in a text cell.
4. **Top-10 tables.** Write one function from lesson 4.3’s pieces and run it on every prompt:

In [ ]:
def top10(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs)
    logits = out.logits[0, -1].double().numpy()
    p = probs(logits)                # probs() from lesson 4.3
    best = np.argsort(p)[::-1][:10]
    return [(tokenizer.decode(int(i)), round(p[i], 3))
            for i in best]